# K-Nearest Neighbors From Scratch

This walkthrough implements Manhattan-distance KNN with NumPy and evaluates it on scikit-learn's bundled handwritten digits. See the README for the separate chest X-ray experiment and the 2022 notebook. Install the repository's requirements and run from the repository root.

## 1. Inspect an 8 × 8 image

Each observation is a small grayscale raster. Flatten the pixels to get a feature vector. Manhattan distance adds the absolute differences between corresponding pixels.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.datasets import load_digits
from evaluate import evaluate
from knn import KNearestNeighbors

digits = load_digits()
print(f'{len(digits.images):,} digit images with shape {digits.images[0].shape}')
print(f'First label: {digits.target[0]}')
print(digits.images[0].astype(int))

## 2. Verify distance and exact-match voting

Cast unsigned pixels to floating point before subtracting. If an inverse-distance vote encounters exact matches, only those matches vote, which handles zero distances directly.

In [ ]:
pixels = np.array([[0], [250]], dtype=np.uint8)
model = KNearestNeighbors(k=1).fit(pixels, ['near', 'far'])
assert model.predict(np.array([[1]], dtype=np.uint8)).tolist() == ['near']

exact = KNearestNeighbors(k=3, weights='distance').fit([[4], [5], [6]], ['match', 'other', 'other'])
assert exact.predict([[4]]).tolist() == ['match']

## 3. Select k on validation data

Reserve a stratified test set before model selection. Use a second split of the training data to compare k = 1 through 10 and both voting methods by macro F1. Refit on all training examples and score once on the untouched test split.

In [ ]:
output = Path('outputs')
result = evaluate(None, output)
display(pd.DataFrame(result['validation_scores']).pivot(index='k', columns='weights', values='macro_f1').round(3))
print('Selected:', result['selection'])
print(f"Test accuracy: {result['test_accuracy']:.3f} | macro F1: {result['test_macro_f1']:.3f}")
display(Image(filename=str(output / 'confusion_matrix.png')))

## 4. Read the errors

The confusion matrix reports which digit pairs share close pixel patterns. KNN has no training gradient and stores its reference examples. Each new image requires a distance calculation against that reference set.